# Which pages keep losing the clicks they should be getting?

**A validated click-shortfall queue for a content portfolio** · Capstone, FlyRank ML Internship, Lane 4 (CTR opportunity scoring)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neilivanorencia/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

## Abstract

Content teams need to know which existing pages are losing clicks they should be getting, and whether that loss will last long enough to be worth fixing. Using FlyRank's warehouse of 78.8 million daily Search Console records across about 50 client sites, each page's clicks were compared with what its exact position and its own site predict, and a page counted as a real shortfall only when it earned at most half of that and a negative binomial test ruled out bad luck. Real shortfalls proved persistent: a page with one this month was 3.8 to 4.8 times as likely to have one next month, including on a sealed month no analysis had touched. A transparent rule (rank by missed clicks) and a logistic regression model found those pages about equally well at the top of the list, 46 to 47 real shortfalls in 50 picks on the sealed month against a base rate of 2.2%, while the model ranked the rest of the list far better and the rule pointed at bigger click pools. The result is a monthly review list, ordered by the rule and labelled with the model's confidence, that a content editor can use as decision support.

**How to read it:** this notebook mirrors the deployed paper section by section. Each section states the finding in words, then a code cell pulls the evidence straight from the saved outputs of the weekly notebooks (`w03` to `w07`), so every number here can be traced to the run that produced it.


## Setup

**Method:** this cell finds the project folder (on Colab, it first clones the repository given in `REPO_URL`) and defines four small helpers: one to open a weekly notebook, one to pull a saved table by its title, one to pull a saved chart, and one to check that the numbers quoted in this notebook appear in the tables they come from. It reads no warehouse data and needs no token.


In [ ]:
import base64
import json
import os
import subprocess
import sys

from IPython.display import Image, Markdown, display

REPO_URL = "https://github.com/neilivanorencia/flyrank-ml-internship"  # the GitHub repository holding this project; used only when the notebooks are not found locally (for example on Colab)

# Locally, walk up to the project's top folder (the one holding work/notebooks); on Colab, clone the repository first.
while not os.path.isdir("work/notebooks") and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir("..")
if not os.path.isdir("work/notebooks") and "google.colab" in sys.modules and REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/project"], check=True)
    os.chdir("/content/project")
assert os.path.isdir("work/notebooks"), "The weekly notebooks were not found; run this from the project folder or set REPO_URL."
NOTEBOOKS = "work/notebooks"
OUTPUTS = "work/outputs"


def notebook(name: str) -> dict:
    """Return a weekly notebook as a dictionary."""
    with open(f"{NOTEBOOKS}/{name}.ipynb", encoding="utf-8") as handle:
        return json.load(handle)


def saved_table(name: str, title: str, rows: int | None = None) -> str:
    """Return the saved Markdown table whose bold title starts with `title`, optionally cut to its first `rows` rows."""
    for cell in notebook(name)["cells"]:
        for output in cell.get("outputs", []):
            text = "".join(output.get("data", {}).get("text/markdown", []))
            if text.startswith(f"**{title}"):
                if rows is not None:
                    lines = text.splitlines()
                    start = next(i for i, line in enumerate(lines) if line.startswith(("| ---", "| :"))) + 1
                    text = "\n".join(lines[:start + rows])
                return text
    raise KeyError(f"{name}: no saved table titled '{title}'")


def saved_chart(name: str, number: int = 0) -> bytes:
    """Return the `number`-th saved chart of a weekly notebook as PNG bytes."""
    charts = [output["data"]["image/png"] for cell in notebook(name)["cells"] for output in cell.get("outputs", [])
              if "image/png" in output.get("data", {})]
    return base64.b64decode(charts[number])


def check(text: str, *values: str) -> None:
    """Fail loudly if a number quoted in this notebook is missing from the table it is quoted from."""
    missing = [value for value in values if value not in text]
    assert not missing, f"Quoted numbers not found in the source table: {missing}"


WEEKLY = {"w01_research_question": "ML-01 to ML-02", "w02_ml_task_framing": "ML-03", "w03_data_contract": "ML-04",
          "w03_feature_leakage_check": "ML-05", "w04_signal_audit": "ML-06", "w04_baseline_score": "ML-07",
          "w05_model": "ML-08", "w06_validation_audit": "ML-09", "w07_action_playbook": "ML-10"}
display(Markdown("**The weekly notebooks this capstone draws on**\n\n| Notebook | Card | Title | Findings |\n| --- | --- | --- | ---: |\n" + "\n".join(
    f"| `{name}` | {card} | {''.join(notebook(name)['cells'][0]['source']).splitlines()[0].lstrip('# ')} | "
    f"{sum(1 for cell in notebook(name)['cells'] if cell['cell_type'] == 'markdown' and ''.join(cell['source']).startswith('**Finding:**'))} |"
    for name, card in WEEKLY.items())))

## 1. Question

**The research question.** _Which existing pages are earning far fewer clicks than their position and their own site predict, beyond what bad luck explains, and will that shortfall still be there next month?_

**The decision it supports.** Every month a content editor at FlyRank has thousands of live pages and time to review a few dozen. The question is where to look first. A page that ranks well but is not clicked is a cheap opportunity: the visibility is already earned, and a better title, description, or page can turn it into traffic. But a page can also miss clicks by chance for a month and recover on its own, and reviewing it wastes the editor's time.

**Why the obvious rule fails.** A common rule flags any page with at least 500 impressions, a position in the top 20, and a click-through rate below 0.5%. On this portfolio the normal click-through rate sits between 0.24% and 0.42% at every position from 1 to 20, below the 0.5% line everywhere, so the rule flags 74% to 88% of all visible pages, including most of the pages clicking exactly as expected. Judging a page needs its own position and its own site as the yardstick, and a test for luck.

**Method:** this cell shows the evidence for the failing rule from the signal audit: the normal click-through rate at each position against the flag's cutoff.


In [ ]:
flag_curve = saved_table("w04_signal_audit", "Normal click-through by position")
display(Markdown(flag_curve))

## 2. Data

**Source.** FlyRank's internship warehouse, release `v20260703`, read with DuckDB directly from Hugging Face (`FlyRank/internship-warehouse`, gated). Four tables: the daily page performance table (78,835,655 rows, one per page per day, Search Console and GA4), the page content table (519,606 pages, keyword research and content type), the client table (104 clients, history start dates), and a 90-day query table that was used only for diagnosis. All identifiers are pseudonymous hashes; no client name, URL, or search query appears anywhere in this work.

**Unit and windows.** One row is one page in one calendar-month pair: features from month M, outcome from month M+1. Three development pairs (February→March, March→April, April→May 2026) built and tuned everything; the May→June pair stayed sealed until a single final test.

**What was excluded, and why.**

- Days with an impossible position: 163,189 March page-days showed position 0, and 101,548 showed a position between 0 and 1, mostly on busy days, which made pages look top-ranked and inflated their expected clicks. Valid positions start at 1.
- Clients whose search history starts after the first day of month M, so every client is measured over a whole month.
- Pages without impressions and a valid position in both months of a pair.
- Pages FlyRank optimized during the pair: someone already acted on them, so the next month cannot judge them.
- From the content table, every field that could postdate month M: word counts and edit dates come from a July 2026 snapshot, and optimization dates and eligibility flags record FlyRank's own later decisions. Only keyword-research fields, fixed before a page is written, are features.

**Method:** this cell shows the four labelled month pairs the work rests on.


In [ ]:
pairs_table = saved_table("w07_action_playbook", "Labelled month pairs")
check(pairs_table, "114,972", "155,707", "157,530", "137,275", "8,465", "10,031", "6,525", "2,978")
display(Markdown(pairs_table))

## 3. Methodology

**Expected clicks.** For each page and month: impressions × the pooled click-through rate of its position bin (each position from 1 to 10, then 11 to 20, 21 to 50, and beyond) × its client's factor (the site's clicks ÷ its expected clicks, +1 on both sides). The yardstick is therefore the page's own position on its own site.

**The label (a real shortfall).** Clicks in a month at most half of expected, _and_ a negative binomial test saying luck explains the gap less than 5% of the time. Clicks vary far more than a Poisson count allows (1.48 times as much for pages with 10 to 29 clicks, 41 times for pages with 1,000 or more), so the spread factor α was estimated by the method of moments on pages whose position barely moved, once per development pair (0.050, 0.032, 0.039), and the median, 0.039, was locked. The target is a real shortfall in month M+1.

**Baseline (rule B).** Missed clicks in month M (expected minus actual), counted only for pages already short by the target's rule in month M. It was chosen from three candidate rules by a criterion fixed in advance and then frozen.

**Model.** Logistic regression on 25 features: month-M traffic (impressions, clicks, click-through, position, its daily spread, days with impressions, expected clicks), rule B's two month-M signals, and keyword research (search volume, competition, cost per click, intent, content type, keyword length). A depth-3 tree, a random forest, and gradient boosting were trained the same way for comparison.

**Validation.** Walk-forward and grouped by client: each test pair is scored by models trained only on earlier pairs and only on other clients, so every test page comes from a month and a site the model never saw. A model beats rule B only if its mean Precision@50 is more than 0.02 higher, or within 0.02 and higher on Capture@50 (the share of next month's missed clicks held by the top 50). The base rate is printed beside every precision. Uncertainty comes from 1,000 client-level bootstrap resamples, and a sealed month (May→June) was scored once at the end.

**Leakage checks.** Scrambling every month M+1 column, the target, and every product field leaves all 25 features unchanged; no feature crosses the 0.90 separation alarm once page size is held fixed; a planted leak lifts the score sharply, so the harness catches leaks; and no test page appears in its model's training rows.

**Method:** this cell shows the leakage audit's summary and how much looser validation designs would have flattered the model.


In [ ]:
leakage = saved_table("w06_validation_audit", "Leakage audit (all checks passed)")
designs = saved_table("w06_validation_audit", "Means over the two test pairs, and how far each design sits above the strictest")
check(designs, "0.920", "0.890")
display(Markdown(leakage))
display(Markdown(designs))

## 4. Results (vs baseline)

**Real shortfalls persist.** Among pages with 1,000 to 9,999 impressions, a page with a real shortfall in one month had one again the next month 28% to 51% of the time, against 6% to 13% for pages without one: 3.8 to 4.8 times as often, on the sealed month too. This is what makes a monthly queue worth having.

**On the development pairs,** logistic regression led rule B on mean Precision@50 (0.890 against 0.800), but its top 50 held fewer of next month's missed clicks (3.59% against 5.60%).

**On the sealed month,** both found real shortfalls at nearly the same rate: Precision@50 of 0.94 for the model against 0.92 for rule B, with a base rate of 2.2%, about 46 to 47 real shortfalls in 50 picks where random picks would find one. The gap of 0.02 sits inside the margin fixed in advance, so Capture@50 decided, and rule B's top 50 held clearly more of June's missed clicks (19.2% against 12.4%). **By the rule fixed before the test, the model did not beat rule B.** The model still ranks the whole list far better (ROC-AUC 0.915 against 0.711).

**How certain.** The model led in all three test sets (+0.06, +0.12, +0.02), but every 95% interval crosses zero; rule B's lead on missed clicks is clear in all three. At the top of the queue the two are level on hits, and rule B reliably finds the bigger click pools.

**Method:** this cell shows the persistence evidence, the development comparison, the sealed test, and the bootstrap intervals, each from the notebook that computed it.


In [ ]:
persistence = saved_table("w06_validation_audit", "Claim 2 evidence")
development = saved_table("w05_model", "Means over the two test pairs, and the verdict")
sealed = saved_table("w06_validation_audit", "The sealed pair")
bootstrap = saved_table("w06_validation_audit", "Logistic regression minus rule B, client-level bootstrap")
check(persistence, "3.9x", "4.5x", "3.8x", "4.8x")
check(development, "0.800", "0.890", "5.60%", "3.59%")
check(sealed, "0.94", "0.92", "19.2%", "12.4%", "0.915", "0.711", "2.2%")
check(bootstrap, "+0.06", "+0.12", "+0.02")
for table in (persistence, development, sealed, bootstrap):
    display(Markdown(table))

## 5. Limitations

- **No measured gain.** The work shows where clicks are missing and that the gap tends to last, not how many clicks a fix recovers. Some shortfalls come from results-page features (ads, AI answers) that no rewrite can beat. Measuring the gain needs an experiment: fix a random half of a month's list and compare it with the other half.
- **Association, not cause.** Persistence and every pattern here are observed in one portfolio over five months; none is a causal claim.
- **Fifty pages per test.** Each Precision@50 rests on 50 pages, so the intervals are wide and no single lead of the model is clear of chance.
- **One sealed month.** It removes the risk of tuning towards the test, but cannot show behaviour across seasons; the position mix has already drifted since February, past the monitoring trigger.
- **Small pages are rarely judged.** A real shortfall needs enough traffic to rule out luck, so the queue reaches a quarter of pages with 10,000 or more impressions but only 0.1% of the smallest.
- **Visibility loss is a different problem.** Most of the model's misses were pages that lost their search visibility the next month, which no click-based score can foresee.
- **Changes after earlier runs.** Gradient boosting's stopping rule and rule B's two features were added after reading earlier development runs; the comparison rule and the sealed test were fixed in advance.

**Method:** this cell shows two of these limits in numbers: who the queue reaches by page size, and the monitoring history with its drift trigger.


In [ ]:
reach = saved_table("w07_action_playbook", "Who the queue reaches, by page size")
monitoring = saved_table("w07_action_playbook", "Monitoring history against the triggers")
check(reach, "25.9%", "0.1%")
check(monitoring, "0.561", "Drift")
display(Markdown(reach))
display(Markdown(monitoring))

## 6. Ranked recommendations

**The playbook.** Once a month, rank every page with a real shortfall by its missed clicks (rule B), take the first 50 with at most 10 from any one client, label each with the model's confidence, and give each one action, chosen in this order:

1. **Check indexing and rankings first** when impressions halved since the previous month: the page is leaving search, and a rewrite will not help.
2. **Wait and re-check next month** when the daily position spread is 5 places or more: these pages repeated least often (19%).
3. **Review site-wide** when the whole site clicks at under half of what its positions predict.
4. **Check the results page first** for buying-intent keywords, where ads and shopping results may take the clicks.
5. **Improve the content** when GA4 shows most visitors leaving without engaging.
6. **Rewrite the title and description** otherwise.

**Evidence that it works.** On the May→June back-test, 16 of the 17 review-list pages FlyRank did not touch (94%) were real shortfalls again in June, and FlyRank's own team optimized 33 of the 50 during those two months. The confidence label orders outcomes correctly (62%, 33%, and 16% repeated for high, medium, and low), though its numbers run 10 to 15 points high. A longer list holds more of the missed clicks at a lower hit rate (30% of June's missed clicks at 500 pages, with 81% hits).

**What a person must check** before acting: the live results page for the main keyword, whether the page is still indexed, and whether it changed since the month the list was built from. Nothing is published, de-indexed, or promised automatically.

**Method:** this cell shows the back-test, the confidence check, and the first twelve pages of the July review list built from June 2026 data.


In [ ]:
tradeoff = saved_table("w07_action_playbook", "Queue size against what it captured in June")
confidence = saved_table("w07_action_playbook", "Confidence label against the June outcome")
july = saved_table("w07_action_playbook", "Review list for July", rows=12)
check(tradeoff, "94%", "81%", "30.3%")
check(confidence, "62%", "33%", "16%")
for table in (tradeoff, confidence, july):
    display(Markdown(table))

## 7. Artifacts the paper embeds

The paper uses three charts, one for each step of its story: the problem (a flat rule misjudges pages), the honest result (model and rule level at the top), and the resolution (a review list sized on purpose).

**Method:** this cell collects the first two charts from the notebooks that drew them and saves them to `work/outputs/` under the names the paper uses. The third is the action playbook's own export, `queue_size_tradeoff.png`, already in `work/outputs/`, so it is shown from there rather than saved twice (from the playbook notebook's saved output if the file is not present).


In [ ]:
FIGURES = {"Flat CTR Rule": ("w04_signal_audit", "A page clicking at the normal rate fails the 0.5% flag at every position from 1 to 20."),
           "Model vs Rule": ("w06_validation_audit", "The model leads rule B in every test set, but no single lead is clear of chance.")}
os.makedirs(OUTPUTS, exist_ok=True)
for number, (file_name, (source, caption)) in enumerate(FIGURES.items(), start=1):
    picture = saved_chart(source)
    with open(f"{OUTPUTS}/{file_name}", "wb") as handle:
        handle.write(picture)
    display(Markdown(f"**Figure {number}: {file_name}** (from `{source}`). {caption}"))
    display(Image(picture, width=900))

# Figure 3 is the action playbook's own export; it is shown, not copied.
PLAYBOOK_FIGURE = f"{OUTPUTS}/queue_size_tradeoff.png"
if os.path.exists(PLAYBOOK_FIGURE):
    with open(PLAYBOOK_FIGURE, "rb") as handle:
        picture = handle.read()
else:
    picture = saved_chart("w07_action_playbook")
display(Markdown("**Figure 3: Queue Size Tradeoff** (exported by `w07_action_playbook`). "
                 "A longer list holds more of the missed clicks at a lower hit rate; 50 pages keeps hits near 94%."))
display(Image(picture, width=900))

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
